# 08 — Taxonomy compression pipeline

Three stages (taxonomy classifier only):

1. **Pruning** — L1 unstructured on classification heads (optional).
2. **Quantization** — dynamic **INT8** on `nn.Linear` + `nn.Conv2d` for faster **CPU** inference.
3. **Huffman** — lossless coding of serialized bytes → smaller **on-disk** file.

**Source checkpoint:** uncomment **one** `SRC` below (match the run you trained in notebooks `03`–`05`). **DST** should be a new `.huff.pt` path next to it.


In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
sys.path.append(str((ROOT / 'src').resolve()))

from fish_ai.compress import (
    TaxonomyCompressionConfig,
    compress_taxonomy_checkpoint,
    load_taxonomy_checkpoint_auto,
    load_taxonomy_for_inference,
)

OUT = ROOT / 'outputs'

# --- Pick one SRC / DST pair (uncomment one block) ---
SRC = (OUT / 'taxonomy_ssl' / 'taxonomy_ssl_resnet50.pt').resolve()
DST = (OUT / 'taxonomy_ssl' / 'taxonomy_ssl_resnet50_compressed.huff.pt').resolve()

# SRC = (OUT / 'federated_fedavg' / 'fedavg_global_model.pt').resolve()
# DST = (OUT / 'federated_fedavg' / 'fedavg_global_model_compressed.huff.pt').resolve()

# SRC = (OUT / 'fewshot_taxonomy' / 'k5_seed0' / 'model.pt').resolve()  # edit k / seed
# DST = SRC.with_name(SRC.stem + '_compressed.huff.pt')

DST.parent.mkdir(parents=True, exist_ok=True)

print('SRC exists:', SRC.exists(), SRC)
print('DST:', DST)


## Huffman codec sanity check

In [ ]:
from fish_ai.compress.huffman_bytes import huffman_encode, huffman_decode

sample = b'fish_ai compression test bytes' * 50
assert huffman_decode(huffman_encode(sample)) == sample
print('roundtrip OK, ratio:', len(huffman_encode(sample)) / len(sample))


## Run pipeline

Tune `TaxonomyCompressionConfig`: `prune_head_l1_amount`, `apply_prune`, `quantize`, `huffman_wrap`.

In [ ]:
if not SRC.exists():
    raise FileNotFoundError(
        'Train a taxonomy model (notebooks 03–05) or set SRC/DST in the cell above. Missing: ' + str(SRC)
    )

cfg = TaxonomyCompressionConfig(
    prune_head_l1_amount=0.20,
    apply_prune=True,
    quantize=True,
    huffman_wrap=True,
)

report = compress_taxonomy_checkpoint(SRC, DST, cfg)
print(report)


## Load compressed checkpoint + smoke forward (CPU)

In [ ]:
import torch

ckpt = load_taxonomy_checkpoint_auto(DST, map_location='cpu')
model = load_taxonomy_for_inference(ckpt)
model.cpu()

x = torch.randn(1, 3, 224, 224)
with torch.inference_mode():
    out = model(x)
print({k: v.shape for k, v in out.items()})
